# ScamLens-AI: Sample LLM Scam Classification Benchmark

This notebook evaluates scam detection against real-world sample messages (`data/sample_data.json`) using the **ScamLens-AI YAML Prompt Catalog** (`text_triage_v1.yaml`) and universal AI client (Groq Llama-3.3, OpenAI, Ollama, etc.).

In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv
from random import sample

# Ensure project root and src are on sys.path
project_root = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Load environment variables from .env
env_file = project_root / ".env"
load_dotenv(env_file) if env_file.exists() else load_dotenv()

from scamlens.ai import get_ai_client, load_prompt, list_available_prompts
from scamlens.core.config import get_settings

settings = get_settings()
client = get_ai_client()

print(f"Project root: {project_root}")
print(f"AI Provider:  {settings.default_ai_provider.upper()}")
print(f"Target Model: {client.model}")
print(f"API Key configured: {bool(client.api_key)}")

Project root: /home/vishnu/ScamLens-AI
AI Provider:  GROQ
Target Model: openai/gpt-oss-20b
API Key configured: True


In [ ]:
# Load sample dataset
candidates = [
    project_root / "data" / "sample_data.json",
    project_root / "data" / "data.json",
]
data_path = next((p for p in candidates if p.is_file()), None)

if not data_path:
    raise FileNotFoundError(f"Dataset not found in candidates: {candidates}")

with open(data_path, "r", encoding="utf-8") as f:
    data_json = json.load(f)

total_rows = len(data_json.get("rows", []))
print(f"Successfully loaded {total_rows} records from: {data_path.relative_to(project_root)}")

Successfully loaded 3787 records from: data/sample_data.json


In [4]:
# List available catalog prompts and load text_triage_v1 template
available = list_available_prompts()
print(f"Available catalog prompts: {available}")

prompt = load_prompt("text_triage_v1")
print(f"\nLoaded prompt template: '{prompt.name}' (v{prompt.version})")
print(f"Description:      {prompt.description}")
print(f"Input variables:  {prompt.input_variables}")
print(f"Model parameters: {prompt.model_parameters}")
print(f"Few-shot pairs:   {len(prompt.few_shot_examples)}")

Available catalog prompts: ['action_guidance_v1', 'evidence_synthesis_v1', 'investigation_agent_v1', 'text_triage_v1', 'vision_screenshot_v1']

Loaded prompt template: 'text_triage_v1' (v1.0.0)
Description:      Rapid screening and feature extraction for SMS, WhatsApp, and Hinglish messages.
Input variables:  ['text']
Model parameters: {'temperature': 0.0, 'max_tokens': 150, 'response_format': 'json_object'}
Few-shot pairs:   3


In [5]:
# Single test inference using rendered prompt messages
sample_idx = 150
sample_text = data_json["rows"][sample_idx]["text"]
sample_label = data_json["rows"][sample_idx].get("label")  # 0 = legitimate, 1 = scam

# Render OpenAI-compatible messages (system + few-shot examples + target text)
messages = prompt.render_messages(text=sample_text)

temperature = prompt.model_parameters.get("temperature", 0.0)
response = client.chat_completion(
    messages=messages,
    temperature=temperature,
)

content = response["choices"][0]["message"].get("content", "")

# Parse JSON or text classification
try:
    parsed = json.loads(content)
    pred_label = 1 if parsed.get("is_suspicious") else 0
    summary = parsed.get("summary", "")
except Exception:
    pred_label = 1 if any(w in content.upper() for w in ["SCAM", "SUSPICIOUS", "TRUE"]) else 0
    summary = content[:80]

print(f"Sample [{sample_idx}]: {sample_text}")
print(f"True Label:      {sample_label} (0=Legitimate, 1=Scam)")
print(f"Predicted Label: {pred_label}")
print(f"Model Summary:   {summary}")

Sample [150]: Update_Now - Xmas Offer! Latest Motorola, SonyEricsson & Nokia & FREE Bluetooth! Double Mins & 1000 Txt on Orange. Call MobileUpd8 on 08000839402 or call2optout/F4Q=
True Label:      1 (0=Legitimate, 1=Scam)
Predicted Label: 1
Model Summary:   Spam text offering free Bluetooth and urging a call to a number that may be a premium‑rate scam, typical of phishing‑style fraud.


In [12]:
# Batch evaluation loop across sampled indices
len_rows = len(data_json["rows"])
indices = sample(range(len_rows), 10)
results = []

print(f"Evaluating {len(indices)} samples with model '{client.model}' and prompt '{prompt.name}'...\n")

for i in indices:
    if i >= len(data_json["rows"]):
        continue
    row = data_json["rows"][i]
    query_text = row["text"]
    ground_truth = row.get("label")
    
    messages = prompt.render_messages(text=query_text)
    response = client.chat_completion(
        messages=messages,
        temperature=prompt.model_parameters.get("temperature", 0.0),
    )
    
    content = response["choices"][0]["message"].get("content", "")
    try:
        parsed = json.loads(content)
        pred_label = 1 if parsed.get("is_suspicious") else 0
        category = parsed.get("scam_category", "UNKNOWN")
    except Exception:
        pred_label = 1 if any(w in content.upper() for w in ["SCAM", "SUSPICIOUS", "TRUE"]) else 0
        category = "RAW_TEXT"
        
    is_correct = (pred_label == ground_truth) if ground_truth is not None else None
    results.append({
        "index": i,
        "text": query_text,
        "ground_truth": ground_truth,
        "pred_label": pred_label,
        "correct": is_correct,
        "category": category,
    })
    
    icon = "✅" if is_correct else "❌"
    preview = (query_text[:45] + "...") if len(query_text) > 45 else query_text
    print(f"[{i:3d}] {icon} Pred: {pred_label} (Cat: {category:12s}) | True: {ground_truth} | Text: {preview}")

# Metrics Summary
total_eval = len(results)
correct_count = sum(1 for r in results if r["correct"])
accuracy = (correct_count / total_eval) * 100 if total_eval > 0 else 0.0

print("-" * 70)
print(f"Batch Summary: {correct_count}/{total_eval} correct ({accuracy:.1f}% accuracy)")
print("-" * 70)

Evaluating 10 samples with model 'openai/gpt-oss-20b' and prompt 'text_triage_v1'...

[3567] ✅ Pred: 0 (Cat: NONE        ) | True: 0 | Text: I don't run away frm u... I walk slowly &amp;...
[1782] ✅ Pred: 0 (Cat: NONE        ) | True: 0 | Text: An excellent thought by a misundrstud frnd: I...
[2977] ✅ Pred: 0 (Cat: NONE        ) | True: 0 | Text: Mmm thats better now i got a roast down me! i...
[612] ✅ Pred: 1 (Cat: PHISHING    ) | True: 1 | Text: RBI ke taraf se aapke bank account me ₹1,00,0...
[1321] ✅ Pred: 0 (Cat: NONE        ) | True: 0 | Text: We are planning a weekend trip to Lonavala. C...
[2918] ✅ Pred: 0 (Cat: NONE        ) | True: 0 | Text: Yes. I come to nyc for audiitions and am tryi...
[3264] ✅ Pred: 0 (Cat: NONE        ) | True: 0 | Text: Ok i wont call or disturb any one. I know all...
[2349] ✅ Pred: 0 (Cat: NONE        ) | True: 0 | Text: Y?WHERE U AT DOGBREATH? ITS JUST SOUNDING LIK...
[3228] ✅ Pred: 0 (Cat: NONE        ) | True: 0 | Text: I can't believe how attached